# Q2 — Does the declared prior have a useful effect?

Hypothesis: a well-calibrated prior helps in low-data settings, its influence decreases as data become more informative, and a misspecified prior is visible in sensitivity curves. The ablation varies only family prior weights.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
while root != root.parent and not (root / 'bayesian_predictive_model_averaging').is_dir():
    root = root.parent
sys.path.insert(0, str(root))
import numpy as np
from sklearn.metrics import log_loss
from bayesian_predictive_model_averaging import FamilyRegistration, default_family_registry
from EXPERIMENTS.common import classification_data, split_data, fit_classifier

In [2]:
X, y = classification_data(seed=11, kind='nonlinear')
X_train, X_test, y_train, y_test = split_data(X, y, seed=11)
base_registry = default_family_registry()
registries = {
    'uniform': base_registry,
    'linear-heavy': tuple(FamilyRegistration(r.adapter, 10.0 if r.adapter.name == 'linear_mixture' else 1.0) for r in base_registry),
    'random-forest-heavy': tuple(FamilyRegistration(r.adapter, 10.0 if r.adapter.name == 'random_forest' else 1.0) for r in base_registry),
}
results = {}
for label, registry in registries.items():
    model = fit_classifier(X_train, y_train, seed=11, family_registry=registry)
    results[label] = {
        'log_loss': log_loss(y_test, model.predict_proba(X_test)),
        'family_mass': model.get_model_masses()['family'],
        'ess_fraction': model.effective_sample_size_fraction_,
    }
results

{'uniform': {'log_loss': 0.6469698953556174,
  'family_mass': {'gaussian_mixture': 0.12686294567649883,
   'knn': 0.5005845822843925,
   'linear_mixture': 0.12396790410089223,
   'mlp': 0.12295674946186509,
   'random_forest': 0.12562781847635143},
  'ess_fraction': 0.9997791197889692},
 'linear-heavy': {'log_loss': 0.6365759924765124,
  'family_mass': {'knn': 0.24989853233331888,
   'linear_mixture': 0.6262870123566162,
   'mlp': 0.12381445531006495},
  'ess_fraction': 0.9999016035497547},
 'random-forest-heavy': {'log_loss': 0.6231138985417831,
  'family_mass': {'gaussian_mixture': 0.12659892631402134,
   'knn': 0.2472446864861147,
   'linear_mixture': 0.12426945032995773,
   'random_forest': 0.5018869368699063},
  'ess_fraction': 0.9998777979185813}}

Repeat the registry sweep over at least 20 data seeds and several training-set sizes. Plot test score and family mass against prior concentration. A useful result should distinguish prior influence from Monte Carlo noise.

## Conclusion from the executed starter run

**Status: not falsified; preliminary support for sensitivity.** Changing only family prior weights changed both posterior family mass and test log loss: 0.6470 for uniform, 0.6366 for linear-heavy, and 0.6231 for random-forest-heavy. The one-seed run does not establish that the best prior is generally helpful, and it does not test the predicted reduction of prior influence with more data.